# RF–NSGA-II Optimization for Catalytic Biomass Pyrolysis
This notebook implements a Random Forest surrogate model with NSGA-II multi-objective optimization to explore catalyst-performance tradeoffs in BTX production.


## Motivation
The catalytic biomass pyrolysis dataset reveals trade-offs between maximizing total yield (Y) and enhancing selectivity to individual aromatics (SB, ST, SX). For instance, catalysts that promote high conversion (e.g., Fe-HZSM) often differ from those maximizing specific BTX components (e.g., Ga-HZSM for SB, Sn-HZSM for SX). This motivated a machine learning–based surrogate optimization framework to systematically identify Pareto-optimal catalyst-condition combinations under process constraints.


In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from pymoo.core.problem import Problem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize
from pymoo.termination import get_termination
from pymoo.operators.sampling.rnd import FloatRandomSampling
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.core.variable import Real, Integer, Choice
import matplotlib.pyplot as plt
import seaborn as sns

## Load and Preprocess Dataset

In [2]:
# Load cleaned dataset with promoter data
df = pd.read_excel('promoter_only.xlsx')
df = df.drop(columns=['X'])
df = df.dropna()
df.head()

,H/C,Promoter,Promoter loading,Ec,Si/Al,Acidity,SBET,Mode,T,CB,WHSV,Y,SB,ST,SX
0,-0.410623,Cu,0.25,3.49,24.0,0.73,322.7,2,500,1.0,12.0,8.85,29.943503,30.621469,20.000000
1,-0.410623,Cu,0.50,3.49,24.0,1.12,316.8,2,500,1.0,12.0,16.99,16.362566,25.485580,31.547969
2,-0.410623,Cu,1.00,3.49,24.0,1.04,310.5,2,500,1.0,12.0,13.06,23.277182,30.398162,26.263400
3,-0.410623,Cu,3.00,3.49,24.0,1.39,306.6,2,500,1.0,12.0,14.18,27.644570,25.811001,27.715092
4,-0.410623,Cu,5.00,3.49,24.0,1.54,280.6,2,500,1.0,12.0,9.39,37.167199,41.853035,6.922258


## Train Random Forest Surrogate Model

In [3]:
X = df.drop(columns=['Promoter','Y', 'SB', 'ST', 'SX'])
y = df[['Y', 'SB', 'ST', 'SX']]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)
model = MultiOutputRegressor(RandomForestRegressor(n_estimators=200, random_state=42))
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
r2_score(y_test, y_pred, multioutput='raw_values')

array([0.50464533, 0.59909879, 0.51057308, 0.58609075])

## Define Optimization Design Space

In [16]:
from pymoo.core.problem import ElementwiseProblem

# Bounds setup with fixed and variable inputs
design_space = {
    'H/C': [0.2, 0.4, 0.6],
    'Promoter loading': [1, 5, 10],
    'Ec': [1.63, 2.81, 3.14, 3.49, 6.25, 6.8, 4.28],
    'Si/Al': [25, 40],
    'Acidity': [0.6, 0.7, 0.8],
    'SBET': [360],  # Fixed
    'Mode': [1,2],  
    'T': [400,500, 600, 650],
    'CB': [2, 5, 10],
    'WHSV': [2, 6, 10]
}

## Run NSGA-II Optimization
This step performs the evolutionary search for Pareto-optimal candidates under the constraints Y ≥ 35% and 60–70% BTX selectivity (SB + ST + SX).


In [26]:
# Variables and bounds
variables = list(design_space.keys())
bounds = [design_space[var] for var in variables]

# Define NSGA-II problem
class CatalystOptimization(ElementwiseProblem):
    def __init__(self):
        super().__init__(n_var=len(variables),
                         n_obj=4,  # Y, SB, ST, SX
                         n_constr=3,
                         xl=np.zeros(len(variables)),
                         xu=np.ones(len(variables)))

    def _evaluate(self, x, out, *args, **kwargs):
        # Map normalized [0,1] -> actual values
        x_real = [bounds[i][int(round(x[i] * (len(bounds[i]) - 1)))] for i in range(len(x))]
        inputs = dict(zip(variables, x_real))

        # Prepare input row
        X_input = np.array([list(inputs.values())])

        # Get per-tree predictions for each output
        preds_per_tree = np.array([
        [est.predict(X_input)[0] for est in output_model.estimators_]
        for output_model in model.estimators_
        ])


        mean_pred = preds_per_tree.mean(axis=1)  # shape = (4,)
        std_pred = preds_per_tree.std(axis=1)    # shape = (4,)

        Y, SB, ST, SX = mean_pred
        s_Y, s_SB, s_ST, s_SX = std_pred

        # Objectives (negate to maximize)
        out["F"] = [-Y, -SB, -ST, -SX]

        # Constraints
        bt_sum = SB + ST + SX
        out["G"] = [
            30 - Y,
            max(0, 55 - bt_sum) + max(0, bt_sum - 75),  22 - SB,
        ]

        # Optionally store uncertainties (can be saved externally)
        out["uncertainty"] = [s_Y, s_SB, s_ST, s_SX]


In [27]:
# Set up NSGA-II algorithm
problem = CatalystOptimization()

algorithm = NSGA2(
    pop_size=200,
    sampling=FloatRandomSampling(),
    crossover=SBX(prob=0.9, eta=15),
    mutation=PM(eta=20),
    eliminate_duplicates=True
)

termination = get_termination("n_gen", 10)

# Run optimization
res = minimize(problem,
               algorithm,
               termination,
               seed=42,
               verbose=True)




n_gen  |  n_eval  | n_nds  |     cv_min    |     cv_avg    |      eps      |   indicator  
     1 |      200 |      9 |  0.000000E+00 |  3.5705006053 |             - |             -
     2 |      400 |     21 |  0.000000E+00 |  1.0927355786 |  0.1946072423 |         ideal
     3 |      600 |     52 |  0.000000E+00 |  0.2533531179 |  0.3572085854 |         ideal
     4 |      800 |     89 |  0.000000E+00 |  0.000000E+00 |  0.1137770014 |         ideal
     5 |     1000 |    140 |  0.000000E+00 |  0.000000E+00 |  0.1562211054 |         ideal
     6 |     1200 |    200 |  0.000000E+00 |  0.000000E+00 |  0.0139649256 |         nadir
     7 |     1400 |    200 |  0.000000E+00 |  0.000000E+00 |  0.0125623577 |         ideal
     8 |     1600 |    200 |  0.000000E+00 |  0.000000E+00 |  0.0230305783 |         ideal
     9 |     1800 |    200 |  0.000000E+00 |  0.000000E+00 |  0.0662028135 |         ideal
    10 |     2000 |    200 |  0.000000E+00 |  0.000000E+00 |  0.0129761244 |         ideal

In [28]:
# Extract performance values
pareto_df = pd.DataFrame(res.F, columns=["-Y", "-SB", "-ST", "-SX"])
pareto_df = -pareto_df
pareto_df.columns = ["Y", "SB", "ST", "SX"]

# Extract uncertainties — robust handling
uncertainties = []
for ind in res.pop:
    if hasattr(ind, 'get'):
        u = ind.get("uncertainty")
        if u is not None:
            uncertainties.append(u)
        else:
            uncertainties.append([None, None, None, None])
    else:
        uncertainties.append([None, None, None, None])

# Build DataFrame from uncertainties
uncertainty_df = pd.DataFrame(uncertainties, columns=["σ_Y", "σ_SB", "σ_ST", "σ_SX"])

# Merge with prediction results
pareto_full = pd.concat([pareto_df, uncertainty_df], axis=1)
pareto_full.head()



,Y,SB,ST,SX,σ_Y,σ_SB,σ_ST,σ_SX
0,30.040145,24.537324,29.035518,20.298382,6.329304,13.136891,4.814123,5.245499
1,31.333194,27.825276,30.029081,13.308235,10.122891,8.359039,7.306195,6.054451
2,32.317238,23.335905,33.051871,17.199246,10.315479,8.783446,5.725698,8.865349
3,34.674648,22.108679,29.164211,23.574517,8.826713,9.597844,3.750308,5.119223
4,31.141546,27.670873,30.680115,12.971644,10.210857,7.916222,5.884794,5.896137


In [29]:
pareto_inputs = pd.DataFrame([[
    bounds[i][int(round(sol[i] * (len(bounds[i]) - 1)))] for i in range(len(variables))
] for sol in res.X], columns=variables)

final_results = pd.concat([pareto_inputs, pareto_full], axis=1)
final_results.tail(15)

,H/C,Promoter loading,Ec,Si/Al,Acidity,SBET,Mode,T,CB,WHSV,Y,SB,ST,SX,σ_Y,σ_SB,σ_ST,σ_SX
185,0.4,1,2.81,25,0.8,360,2,650,5,6,31.182145,23.492619,30.270587,20.024129,8.591257,10.277698,3.619277,6.655571
186,0.4,1,2.81,25,0.7,360,1,650,5,10,30.884050,23.343262,30.167412,20.656668,8.417993,9.800572,3.557877,6.290721
187,0.2,1,6.80,40,0.7,360,1,650,10,2,32.599973,22.573088,31.826347,19.195395,9.900839,7.936895,5.573689,8.678211
188,0.4,5,3.14,40,0.7,360,1,650,2,6,32.147358,22.654511,29.213467,23.027430,7.972188,13.415273,4.615058,5.404132
189,0.4,10,2.81,40,0.8,360,1,650,5,10,32.665228,22.785265,28.760634,23.410746,8.359368,9.795942,4.393844,5.482581
190,0.2,10,6.25,25,0.8,360,2,650,2,10,30.788113,25.991465,30.795086,17.825544,9.856665,9.576977,6.370476,8.726047
191,0.4,5,3.14,25,0.7,360,2,650,10,6,32.944884,23.212520,29.577117,21.791846,8.772013,10.023590,3.580013,6.628674
192,0.4,10,3.14,40,0.8,360,2,650,2,2,34.001761,25.078069,27.672971,21.881490,8.477737,13.131621,5.935072,6.257968
193,0.6,5,6.25,40,0.8,360,2,650,5,6,32.110893,24.234206,32.248858,18.251507,8.549219,10.257615,5.011993,8.476746
194,0.4,5,3.49,40,0.7,360,1,650,5,2,34.677955,22.187962,29.159778,23.429064,8.205416,10.342898,4.315873,4.702363


In [30]:
final_results.to_excel("Pareto_solution_NSGAfinal_SB25.xlsx")

In [11]:
# Total BTX selectivity
pareto_full["BTX_total"] = pareto_full["SB"] + pareto_full["ST"] + pareto_full["SX"]

# Filter feasible solutions
feasible_df = pareto_full[
    (pareto_full["Y"] >= 35) &
    (pareto_full["BTX_total"] >= 60) &
    (pareto_full["BTX_total"] <= 70)
]

n_feasible = len(feasible_df)
print(f"Feasible solutions: {n_feasible}")


Feasible solutions: 200


In [12]:
# Total Pareto-optimal solutions found
n_pareto = len(res.opt)
print(f"Total Pareto-optimal (non-dominated) solutions: {n_pareto}")

Total Pareto-optimal (non-dominated) solutions: 200
